In [ ]:
# Website
#    ↓
# Load Content
#    ↓
# Split into chunks
#    ↓
# Convert chunks into embeddings
#    ↓
# Store in FAISS vector DB
#    ↓
# User asks question
#    ↓
# Retriever finds relevant chunks
#    ↓
# LLM gets:
#     - Question
#     - Retrieved chunks
#    ↓
# Final Answer

In [132]:
from langchain.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OllamaEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOllama

url = "https://www.cricbuzz.com/profiles/1413/virat-kohli"

# Reads content from a webpage
loader = WebBaseLoader(url)
docs = loader.load()

# Splits huge text into smaller chunks
# Embedding entire thing is bad because:
# loses meaning, poor retrieval, too large
# So we split into chunks.
# each chunk contains ~1000 characters
# chunks share 200 chars
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

documents = splitter.split_documents(docs)

# Converts text into numerical vectors
# Embeddings are numerical representations of text.
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

# FAISS is a vector Database
# Stores embeddings efficiently.
vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

# Retriever searches vector DB.
# top 3 most similar chunks
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

# Runs local LLM using Ollama
llm = ChatOllama(
    model="gpt-oss:120b-cloud",
    temperature=0
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever
)



In [133]:
query = "What is the height of kohli?"

result = qa.run(query)

print(result)

Virat Kohli is listed at roughly 5 feet 9 inches tall, which is about 175 cm.


In [135]:
query = "highest ODI score of kohli"

result = qa.run(query)

print(result)

Virat Kohli’s highest individual score in a One‑Day International is **183 * (not out)**, which he made against Pakistan during the 2012 Asia Cup in Dhaka.


In [139]:
query = "Batting Career Summary"

result = qa.run(query)

print(result)

**Virat Kohli – Batting Career Summary**

| Format | Matches | Innings | Runs | Balls | Highest | Average | Strike‑Rate | Not Outs | Fours | Sixes |
|--------|--------:|--------:|-----:|------:|--------:|--------:|------------:|---------:|------:|------:|
| **Test** | 123 | 210 | 9 230 | 16 608 | 254 | 46.85 | 55.58 | 13 | 102 | 30 |
| **ODI** | 311 | 299 | 14 797 | 1 577 | 183 | 58.72 | 93.83 | 47 | 713 | 169 |
| **T20I** | 125 | 117 | 4 188 | 1 305 | 122 | 48.74 | 137.05 | 31 | 763 | 124 |
| **IPL** | 282 | 274 | 261 | 6 688 | 113 | 0.09 * | 34.53 | 43 | 698 | 313 |

\*The source lists the IPL average as **0.09**, which is almost certainly a data‑entry error; a realistic IPL average for Kohli is far higher.  

*Key points*  
- Kohli has played **123 Tests**, scoring **9 230 runs** at an average of **46.85**.  
- In **311 ODIs**, he has amassed **14 797 runs** at an impressive **58.72** average.  
- His **125 T20I** outings have yielded **4 188 runs** with a striking **137.05** strike